Plantas

In [0]:
!pip install pydataxm

In [0]:
from pydataxm.pydatasimem import ReadSIMEM, CatalogSIMEM

In [0]:
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

from pydataxm.pydatasimem import ReadSIMEM


bronze_table = "observatorio_dev.bronze.plantas"
historical_start_date = date(2026, 1, 1)
lookback_days = 45

fecha_fin = datetime.now(
    ZoneInfo("America/Bogota")
).date()

bronze_is_empty = len(
    spark.table(bronze_table).head(1)
) == 0

if bronze_is_empty:
    fecha_inicio = historical_start_date
    execution_mode = "BACKFILL"
else:
    fecha_inicio = fecha_fin - timedelta(days=lookback_days)
    execution_mode = "INCREMENTAL"

fecha_inicio_str = fecha_inicio.strftime("%Y-%m-%d")
fecha_fin_str = fecha_fin.strftime("%Y-%m-%d")

print("Modo de ejecución:", execution_mode)
print("Tabla Bronze:", bronze_table)
print(
    f"Rango solicitado a SIMEM: "
    f"{fecha_inicio_str} a {fecha_fin_str}"
)

df_plantas = ReadSIMEM(
    "0bfc9d",
    fecha_inicio_str,
    fecha_fin_str
).main(filter=False)

if df_plantas is None or df_plantas.empty:
    raise ValueError(
        "SIMEM no devolvió plantas para el rango solicitado"
    )

print(f"Registros descargados: {len(df_plantas):,}")

df_plantas.to_json(
    "/Volumes/observatorio_dev/landing/raw_files/plantas.json",
    orient="records",
    lines=True,
    mode="w"
)

In [0]:
# Guardar como un solo archivo JSON usando pandas, sobrescribiendo si ya existe
df_plantas.to_json("/Volumes/observatorio_dev/landing/raw_files/plantas.json", orient='records', lines=True, mode='w')